# FarmSense — Notebook Kaggle
## Gemma 4 Good Hackathon | Kaggle x Google DeepMind

Une seule cellule a executer. Elle installe ce qui manque, demarre Ollama,
telecharge Gemma 4 si necessaire, et genere un lien public via ngrok.

Avant de lancer : remplace TON_TOKEN_NGROK par ton token sur https://dashboard.ngrok.com


In [ ]:
import os, subprocess, time, requests, sys, threading

GITHUB_URL  = 'https://github.com/TON_USERNAME/farmsense'
NGROK_TOKEN = 'TON_TOKEN_NGROK'
MODEL       = 'gemma4:e4b'

def cmd(c): os.system(c)

# 1. Dependances Python
try:
    import flask, gtts, pyngrok
    print('OK Packages deja installes')
except ImportError:
    print('Installation des packages...')
    cmd('pip install -q flask gtts Pillow requests pyngrok')
    print('OK Packages installes')

# 2. Ollama
if not os.path.exists('/usr/local/bin/ollama'):
    print('Installation de Ollama...')
    cmd('apt-get install -y zstd > /dev/null 2>&1')
    cmd('curl -fsSL https://ollama.com/install.sh | sh')
    print('OK Ollama installe')
else:
    print('OK Ollama deja installe')

# 3. Demarrer Ollama
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('Demarrage Ollama...')
for i in range(15):
    time.sleep(3)
    try:
        if requests.get('http://localhost:11434/api/tags', timeout=3).status_code == 200:
            print('OK Ollama pret')
            break
    except:
        print(f'  Attente... {(i+1)*3}s')

# 4. Telecharger Gemma 4 si absent
models = requests.get('http://localhost:11434/api/tags').json()
noms   = [m['name'] for m in models.get('models', [])]
if not any('gemma4' in n for n in noms):
    print(f'Telechargement de {MODEL}...')
    cmd(f'ollama pull {MODEL}')
    print('OK Modele telecharge')
else:
    print(f'OK Modele deja present : {noms}')

# 5. Gardien Ollama
def keep_ollama_alive():
    while True:
        time.sleep(30)
        try:
            requests.get('http://localhost:11434/api/tags', timeout=3)
        except:
            try:
                subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                time.sleep(8)
            except: pass

threading.Thread(target=keep_ollama_alive, daemon=True).start()
print('OK Gardien Ollama actif')

# 6. Code FarmSense
if not os.path.exists('/kaggle/working/farmsense'):
    print('Clonage du repo...')
    cmd(f'git clone {GITHUB_URL} /kaggle/working/farmsense')
else:
    print('Mise a jour du code...')
    cmd('git -C /kaggle/working/farmsense pull')

# 7. Tunnel ngrok
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()
public_url = ngrok.connect(5000)
print(f'LIEN PUBLIC FARMSENSE : {public_url}')
print('Copiez ce lien pour la soumission Kaggle')

# 8. Lancer FarmSense
os.environ['GEMMA_MODEL'] = MODEL
sys.path.insert(0, '/kaggle/working/farmsense/app')
os.chdir('/kaggle/working/farmsense/app')
print('Lancement de FarmSense...')
cmd('python app_flask.py')
